# Q1 — HCD APR (Tables A / A2 / B) from v2

**One input: a v2 database path.** Reads v2 + `housing_rules` only — never `berkeley.db`, never the CKAN mirror (the mirror is the column-schema target, validated against, never a data source). Run it on canonical-v2 **or** a rebuilt-v2 for a comparable APR — that is the Q2 acceptance test.

**A2** is the core: the HCD *Annual Building Activity* matrix — per project, income-tier (Acutely-low → Above-mod) × **DR/NDR** (deed-restricted / non) × milestone (Entitlement / BP / CO), populated for the milestone(s) reached in the report year. Columns match the official HCD `table_a2` schema **exactly (69/69)**.

**Honest data state (surfaced, not hidden):** DR/NDR restriction detail covers **0% of completions** → completion affordable units default to NDR (flagged in `NOTES`); income-tier coverage ~36% → uncategorized `total_units` default to ABOVE_MOD (flagged); `ACUTELY_LOW`/`FIN_ASSIST_NAME`/demo-detail absent in v2 → blank-with-provenance. The structure is HCD-faithful; the cells carry what v2 actually knows.

The logic lives in the tested module `scripts/apr_hcd.py` (smoke-validated: exact column match + matrix reconciliation). This notebook is the linear run-all wrapper.

In [ ]:
# --- PARAMETERS (the only inputs) ---
DB_PATH = '../databases/berkeley_housing_v2.db'   # point at ANY v2 (canonical or rebuilt)
REPORT_YEAR = 2025
CITY = 'Berkeley'                                  # per-city config: RHNA allocations swap, form does not

import sys, sqlite3, pandas as pd
sys.path.insert(0, '../scripts')
import apr_hcd

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True); conn.row_factory = sqlite3.Row
cfg = apr_hcd.BERKELEY   # swap for OAKLAND etc. — same form, different RHNA allocations
print('v2:', DB_PATH, '| year:', REPORT_YEAR, '| city:', cfg['JURIS_NAME'])

## Headline / cumulative block
Completions count, net-new-CO tiles, all-time CO, RHNA 6th-cycle credit (first-BP ≥ projection start, MIN-not-MAX, UC-excluded).

In [ ]:
cumulative = apr_hcd.cumulative_block(conn, cfg)
import json; print(json.dumps(cumulative, indent=2))

## Table A2 — Annual Building Activity (the matrix, 69 HCD columns)

In [ ]:
a2_rows, a2_provenance = apr_hcd.build_table_a2(conn, REPORT_YEAR, cfg)
table_a2 = pd.DataFrame(a2_rows)
print(f'Table A2 {REPORT_YEAR}: {len(table_a2)} project rows, {len(table_a2.columns)} HCD columns, '
      f'{len(a2_provenance)} provenance-flagged')
table_a2.to_csv(f'table_a2_{REPORT_YEAR}_hcd.csv', index=False)
table_a2.head(10)

## Table A — Entitlements & Table B — RHNA progress

In [ ]:
table_a = pd.DataFrame(apr_hcd.build_table_a(conn, REPORT_YEAR, cfg))
table_a.to_csv(f'table_a_{REPORT_YEAR}_hcd.csv', index=False)
table_b = apr_hcd.build_table_b(conn, cfg)
print('Table A rows:', len(table_a)); print('Table B (RHNA progress):'); print(json.dumps(table_b, indent=2))
table_a.head(10)

## Validation — exact column match to the official HCD schema + matrix reconciliation
(the mirror is used here ONLY as the column-schema target, never as data)

In [ ]:
mirror_cols = [c[1] for c in sqlite3.connect('../databases/hcd_apr_mirror.db').execute('PRAGMA table_info(table_a2)')]
ours = list(table_a2.columns)
missing = [c for c in mirror_cols if c not in ours]; extra = [c for c in ours if c not in mirror_cols]
print(f'A2 column match vs HCD mirror: {len(ours)}/{len(mirror_cols)}  exact={not missing and not extra}')
if missing or extra: print('  missing:', missing, '| extra:', extra)

# matrix conserves units: sum CO cells across all years == all-time CO
co_total = 0
for yr in range(2018, 2027):
    for r, _ in [apr_hcd.build_table_a2(conn, yr, cfg)][:1]:
        for row in r:
            for k, v in row.items():
                if k.startswith('CO_') and (k.endswith('_DR') or k.endswith('_NDR') or k == 'CO_ABOVE_MOD_INCOME') and isinstance(v, (int, float)):
                    co_total += v
print(f'matrix CO-cells total = {co_total} vs all-time CO = {cumulative["all_time_co_units"]}  reconciles={co_total == cumulative["all_time_co_units"]}')